In [6]:
import math
import numpy as np
from typing import Callable, List, Tuple

1.1. Локализация корней

In [ ]:
def localize_roots(f, a, b, step):
    intervals = [] # сюда будем сохранять интервалы, где найден возможный корень
    x = a  # начинаем движение от левой границы

    while x < b:
        x_next = min(x + step, b) # следующая точка, но не выходим за предел b
        # если функция меняет знак, то на этом отрезке может быть корень
        if f(x) * f(x_next) <= 0:
            intervals.append((x, x_next))
        x = x_next # переходим к следующему маленькому отрезку

    return intervals
def f1(x):
    return x**3 - x - 2

intervals = localize_roots(f1, 1, 2, 0.1)
print("Найденные интервалы с возможным корнем:", intervals)

Найденные интервалы с возможным корнем: [(1.5000000000000004, 1.6000000000000005)]


Этот блок ищет отрезки, на которых может находиться корень уравнения.
Идея метода простая: если на концах маленького отрезка функция меняет знак, значит внутри этого отрезка, скорее всего, есть корень.
Мы последовательно двигаемся от точки a до точки b с шагом step и сохраняем подходящие интервалы.

1.2. Метод половинного деления

In [ ]:
def bisection(f, a, b, eps=1e-6, max_iter=1000):
  # проверяем, что на концах интервала функция имеет разные знаки
    if f(a) * f(b) > 0:
        raise ValueError("На концах интервала функция должна иметь разные знаки.")

    history = [] # сюда сохраняем шаги метода

    for i in range(max_iter):
        c = (a + b) / 2 # середина текущего интервала
        fc = f(c)
        history.append((i + 1, a, b, c, fc))

        # если значение функции близко к нулю или интервал уже очень маленький, заканчиваем
        if abs(fc) < eps or (b - a) / 2 < eps:
            return c, history


        # выбираем ту половину интервала, где происходит смена знака
        if f(a) * fc < 0:
            b = c
        else:
            a = c

    return c, history
# Пример данных
def f2(x):
    return x**3 - x - 2

root, hist = bisection(f2, 1, 2)

print("Корень:", root)
print("Проверка f(root):", f2(root))
print("Первые 5 шагов:")
for row in hist[:5]:
    print(row)

Корень: 1.5213804244995117
Проверка f(root): 4.265829404825894e-06
Первые 5 шагов:
(1, 1, 2, 1.5, -0.125)
(2, 1.5, 2, 1.75, 1.609375)
(3, 1.5, 1.75, 1.625, 0.666015625)
(4, 1.5, 1.625, 1.5625, 0.252197265625)
(5, 1.5, 1.5625, 1.53125, 0.059112548828125)


Метод половинного деления на каждом шаге делит текущий отрезок пополам.
Потом определяется, в какой половине находится корень, и работа продолжается только с этой частью.
Так длина отрезка постепенно уменьшается, и приближение к корню становится всё точнее.

1.3. Метод простых итераций

In [ ]:
def simple_iteration(phi, x0, eps=1e-6, max_iter=1000):
    history = [] # история шагов
    x = x0 # начальное приближение

    for i in range(max_iter):
        x_new = phi(x) # считаем новое значение по итерационной формуле
        history.append((i + 1, x, x_new, abs(x_new - x)))


        # если соседние приближения почти одинаковые, завершаем
        if abs(x_new - x) < eps:
            return x_new, history

        x = x_new # переходим к следующему шагу

    return x, history

    # Пример данных
def phi3(x):
    return (x + 2) ** (1 / 3)

root, hist = simple_iteration(phi3, 1.5)

print("Корень:", root)
print("Первые 5 шагов:")
for row in hist[:5]:
    print(row)

Корень: 1.5213796792714414
Первые 5 шагов:
(1, 1.5, 1.5182944859378311, 0.018294485937831118)
(2, 1.5182944859378311, 1.520935263216892, 0.0026407772790608686)
(3, 1.520935263216892, 1.52131569819048, 0.0003804349735880841)
(4, 1.52131569819048, 1.5213704886377613, 5.479044728118332e-05)
(5, 1.5213704886377613, 1.521378379262021, 7.890624259765389e-06)


В методе простых итераций новое значение вычисляется по формуле x_(n+1) = phi(x_n).
То есть каждое следующее приближение строится из предыдущего.
Если разница между соседними приближениями становится меньше заданной точности eps, процесс останавливается.

1.4. Метод Ньютона

In [ ]:
def newton(f, df, x0, eps=1e-6, max_iter=1000):
    history = []
    x = x0

    for i in range(max_iter):
        fx = f(x) # значение функции в текущей точке
        dfx = df(x) # значение производной в текущей точке


        # защита от деления на очень маленькое число
        if abs(dfx) < 1e-14:
            raise ZeroDivisionError("Производная слишком близка к нулю.")


        # основная формула метода Ньютона
        x_new = x - fx / dfx
        history.append((i + 1, x, fx, dfx, x_new, abs(x_new - x)))


        # если изменение стало маленьким, значит решение найдено
        if abs(x_new - x) < eps:
            return x_new, history

        x = x_new

    return x, history

# Пример данных
def f4(x):
    return x**3 - x - 2

def df4(x):
    return 3*x**2 - 1

root, hist = newton(f4, df4, 1.5)

print("Корень:", root)
print("Проверка f(root):", f4(root))
print("Все шаги метода:")
for row in hist:
    print(row)

Корень: 1.5213797068045751
Проверка f(root): 4.529709940470639e-14
Все шаги метода:
(1, 1.5, -0.125, 5.75, 1.5217391304347827, 0.021739130434782705)
(2, 1.5217391304347827, 0.0021369277554046384, 5.947069943289225, 1.5213798059647863, 0.00035932446999642487)
(3, 1.5213798059647863, 5.893874259754739e-07, 5.943789541992352, 1.5213797068045751, 9.916021115330409e-08)


Метод Ньютона использует значение функции и её производной.
На каждом шаге строится новое приближение по формуле x_new = x - f(x)/f'(x).
Этот метод обычно сходится быстро, если начальная точка выбрана удачно.

1.5. Модифицированный метод Ньютона

In [ ]:
def modified_newton(f, df, x0, eps=1e-6, max_iter=1000):
    history = []
    x = x0
    dfx0 = df(x0) # производную считаем только один раз в начальной точке

    if abs(dfx0) < 1e-14:
        raise ZeroDivisionError("Производная в начальной точке слишком близка к нулю.")

    for i in range(max_iter):
        fx = f(x) # текущее значение функции

        # здесь используется одна и та же производная dfx0
        x_new = x - fx / dfx0
        history.append((i + 1, x, fx, dfx0, x_new, abs(x_new - x)))

        if abs(x_new - x) < eps:
            return x_new, history

        x = x_new

    return x, history
# Пример данных
def f5(x):
    return x**3 - x - 2

def df5(x):
    return 3*x**2 - 1

root, hist = modified_newton(f5, df5, 1.5)

print("Корень:", root)
print("Проверка f(root):", f5(root))
print("Первые 5 шагов:")
for row in hist[:5]:
    print(row)

Корень: 1.5213796929329069
Проверка f(root): -8.245021843045208e-08
Первые 5 шагов:
(1, 1.5, -0.125, 5.75, 1.5217391304347827, 0.021739130434782705)
(2, 1.5217391304347827, 0.0021369277554046384, 5.75, 1.521367490825147, 0.00037163960963559894)
(3, 1.521367490825147, -7.260851856183415e-05, 5.75, 1.5213801183935927, 1.2627568445555681e-05)
(4, 1.5213801183935927, 2.446398943867223e-06, 5.75, 1.5213796929329069, 4.254606857934107e-07)


В этом методе производная вычисляется только один раз в начальной точке.
Дальше на всех шагах используется одно и то же значение производной.
Это уменьшает вычислительную нагрузку, но иногда может снизить точность или скорость сходимости.

1.6. Метод секущих

In [ ]:
def secant(f, x0, x1, eps=1e-6, max_iter=1000):
    history = []

    for i in range(max_iter):
        f0 = f(x0) # значение функции в первой точке
        f1 = f(x1) # значение функции во второй точке


        # защита от деления на ноль
        if abs(f1 - f0) < 1e-14:
            raise ZeroDivisionError("Разность значений функции слишком мала.")


        # формула метода секущих
        x2 = x1 - f1 * (x1 - x0) / (f1 - f0)
        history.append((i + 1, x0, x1, x2, f(x2), abs(x2 - x1)))


        # условие остановки
        if abs(x2 - x1) < eps:
            return x2, history


        # сдвигаем точки для следующего шага
        x0, x1 = x1, x2

    return x2, history

# Пример данных
def f6(x):
    return x**3 - x - 2

root, hist = secant(f6, 1, 2)

print("Корень:", root)
print("Проверка f(root):", f6(root))
print("Первые 5 шагов:")
for row in hist[:5]:
    print(row)

Корень: 1.5213797068045645
Проверка f(root): -1.84297022087776e-14
Первые 5 шагов:
(1, 1, 2, 1.3333333333333335, -0.9629629629629624, 0.6666666666666665)
(2, 2, 1.3333333333333335, 1.462686567164179, -0.3333388747951045, 0.12935323383084563)
(3, 1.3333333333333335, 1.462686567164179, 1.5311694321412044, 0.05862641770944954, 0.06848286497702527)
(4, 1.462686567164179, 1.5311694321412044, 1.5209264205152802, -0.0026933002019968733, 0.010243011625924225)
(5, 1.5311694321412044, 1.5209264205152802, 1.5213763166697438, -2.0150192387102805e-05, 0.00044989615446366926)


Метод секущих похож на метод Ньютона, но здесь не нужна производная.
Вместо неё используется прямая, проведённая через две последние точки графика функции.
Точка пересечения этой прямой с осью x даёт новое приближение к корню.

1.7. Метод хорд с неподвижным концом

In [1]:
def fixed_chord(f, x_fixed, x0, eps=1e-6, max_iter=1000):
    history = []
    f_fixed = f(x_fixed) # значение функции в неподвижной точке

    for i in range(max_iter):
        fx0 = f(x0)

        if abs(fx0 - f_fixed) < 1e-14:
            raise ZeroDivisionError("Разность значений функции слишком мала.")
        # строим хорду между неподвижной точкой и текущей точкой
        x_new = x0 - fx0 * (x0 - x_fixed) / (fx0 - f_fixed)
        history.append((i + 1, x0, x_new, f(x_new), abs(x_new - x0)))

        if abs(x_new - x0) < eps:
            return x_new, history

        x0 = x_new # обновляем только подвижную точку

    return x0, history
# Пример данных
def f7(x):
    return x**3 - x - 2

root, hist = fixed_chord(f7, 1, 2)

print("Корень:", root)
print("Проверка f(root):", f7(root))
print("Первые 5 шагов:")
for row in hist[:5]:
    print(row)

Корень: 1.521379923773018
Проверка f(root): 1.2896148247065753e-06
Первые 5 шагов:
(1, 2, 1.3333333333333335, -0.9629629629629624, 0.6666666666666665)
(2, 1.3333333333333335, 1.6428571428571428, 0.7911807580174925, 0.3095238095238093)
(3, 1.6428571428571428, 1.4606345475910694, -0.34443897879037144, 0.18222259526607343)
(4, 1.4606345475910694, 1.5564694284170917, 0.21422886646114936, 0.09583488082602232)
(5, 1.5564694284170917, 1.502630452385402, -0.10984374382421969, 0.053838976031689745)


В этом методе один конец хорды фиксируется, а второй движется.
На каждом шаге строится хорда между фиксированной точкой и текущим приближением.
Затем определяется новая точка пересечения хорды с осью x.

1.8. Метод ложного положения

In [2]:
def false_position(f, a, b, eps=1e-6, max_iter=1000):
    # проверяем наличие смены знака на концах
    if f(a) * f(b) > 0:
        raise ValueError("На концах интервала функция должна иметь разные знаки.")

    history = []

    for i in range(max_iter):
        fa = f(a)
        fb = f(b)
        # находим точку пересечения хорды с осью x
        c = a - fa * (b - a) / (fb - fa)
        fc = f(c)
        history.append((i + 1, a, b, c, fc))
        # если попали достаточно близко к корню, останавливаемся
        if abs(fc) < eps or abs(b - a) < eps:
            return c, history
        # сохраняем только тот интервал, где есть смена знака
        if fa * fc < 0:
            b = c
        else:
            a = c

    return c, history
# Пример данных
def f8(x):
    return x**3 - x - 2

root, hist = false_position(f8, 1, 2)

print("Корень:", root)
print("Проверка f(root):", f8(root))
print("Первые 5 шагов:")
for row in hist[:5]:
    print(row)

Корень: 1.5213796360454928
Проверка f(root): -4.205769621457023e-07
Первые 5 шагов:
(1, 1, 2, 1.3333333333333333, -0.9629629629629635)
(2, 1.3333333333333333, 2, 1.462686567164179, -0.3333388747951045)
(3, 1.462686567164179, 2, 1.5040190039499488, -0.10181797660389025)
(4, 1.5040190039499488, 2, 1.516330564760263, -0.02989480441927217)
(5, 1.516330564760263, 2, 1.519918550023356, -0.00867506585052058)


Метод ложного положения сочетает идею хорд и контроль интервала, в котором находится корень.
Новая точка находится как пересечение хорды с осью x.
После этого сохраняется только тот подынтервал, где функция меняет знак.

1.9. Метод Стэффенсена

In [4]:
def steffensen(phi, x0, eps=1e-6, max_iter=1000):
    history = []
    x = x0

    for i in range(max_iter):
        x1 = phi(x)   # первое приближение по функции phi
        x2 = phi(x1)  # второе приближение по функции phi

        # знаменатель ускоряющей формулы
        denom = x2 - 2 * x1 + x

        # если знаменатель слишком мал, метод дальше применять нельзя
        if abs(denom) < 1e-12:
            print("Остановка: знаменатель стал слишком мал.")
            return x, history

        # формула Стэффенсена
        x_new = x - (x1 - x) ** 2 / denom

        # сохраняем шаг: номер шага, старое значение, новое значение, разницу
        history.append((i + 1, x, x_new, abs(x_new - x)))

        # если новое и старое значения почти совпали, завершаем
        if abs(x_new - x) < eps:
            return x_new, history

        x = x_new

    return x, history


# Пример данных
def phi9(x):
    return (x + 2) ** (1/3)

root, hist = steffensen(phi9, 1.5)

print("Корень:", root)
print("Количество шагов:", len(hist))
print("Первые 5 шагов:")
for row in hist[:5]:
    print(row)

Остановка: знаменатель стал слишком мал.
Корень: 1.52137970680457
Количество шагов: 2
Первые 5 шагов:
(1, 1.5, 1.521380761775069, 0.02138076177506898)
(2, 1.521380761775069, 1.52137970680457, 1.0549704989593067e-06)


Метод Стэффенсена ускоряет обычный метод простых итераций.
Для этого используется дополнительное преобразование, которое позволяет быстрее получить точное значение корня.
Метод удобен, когда есть итерационная функция phi(x).

2.1. Метод Гаусса

In [7]:
def gauss_method(A, b):
    A = np.array(A, dtype=float)   # переводим матрицу в массив numpy
    b = np.array(b, dtype=float)   # переводим правую часть в массив numpy
    n = len(b)

    # Прямой ход метода Гаусса
    for i in range(n):
        # ищем строку с максимальным элементом в текущем столбце
        max_row = np.argmax(abs(A[i:, i])) + i

        # меняем строки местами
        A[[i, max_row]] = A[[max_row, i]]
        b[[i, max_row]] = b[[max_row, i]]

        # зануляем элементы ниже главного
        for j in range(i + 1, n):
            factor = A[j, i] / A[i, i]
            A[j, i:] -= factor * A[i, i:]
            b[j] -= factor * b[i]

    # Обратный ход
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (b[i] - np.dot(A[i, i + 1:], x[i + 1:])) / A[i, i]

    return x


# Пример данных
A = [[3, 2, -1],
     [2, -2, 4],
     [-1, 0.5, -1]]

b = [1, -2, 0]

x = gauss_method(A, b)
print("Решение системы:", x)

Решение системы: [ 1. -2. -2.]


Метод Гаусса решает систему линейных уравнений в два этапа.
Сначала матрица приводится к верхнетреугольному виду, а затем находится решение обратным ходом.
Выбор главного элемента помогает избежать ошибок деления на очень маленькие числа.

2.2. Метод прогонки

In [8]:
def thomas_method(a, b, c, d):
    n = len(d)

    c_prime = np.zeros(n)   # вспомогательные коэффициенты для верхней диагонали
    d_prime = np.zeros(n)   # вспомогательные коэффициенты для правой части

    # первый шаг прогонки
    c_prime[0] = c[0] / b[0]
    d_prime[0] = d[0] / b[0]

    # прямой ход
    for i in range(1, n):
        denom = b[i] - a[i] * c_prime[i - 1]   # знаменатель текущего шага

        if i < n - 1:
            c_prime[i] = c[i] / denom

        d_prime[i] = (d[i] - a[i] * d_prime[i - 1]) / denom

    # обратный ход
    x = np.zeros(n)
    x[-1] = d_prime[-1]

    for i in range(n - 2, -1, -1):
        x[i] = d_prime[i] - c_prime[i] * x[i + 1]

    return x


# Пример данных
# a - нижняя диагональ, b - главная, c - верхняя, d - правая часть
a = [0, 1, 1, 1]
b = [4, 4, 4, 4]
c = [1, 1, 1, 0]
d = [5, 6, 6, 5]

x = thomas_method(a, b, c, d)
print("Решение системы:", x)

Решение системы: [1. 1. 1. 1.]


Метод прогонки применяется для трёхдиагональных систем.
Он работает быстрее обычного метода Гаусса, потому что использует специальную структуру матрицы.
Сначала вычисляются вспомогательные коэффициенты, а потом по ним находится решение.

2.3. Метод простой итерации для СЛАУ

In [9]:
def jacobi_method(A, b, eps=1e-6, max_iter=1000):
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    n = len(b)

    x = np.zeros(n)   # начальное приближение
    history = []

    for k in range(max_iter):
        x_new = np.zeros(n)

        for i in range(n):
            # считаем сумму без диагонального элемента
            s = np.dot(A[i, :], x) - A[i, i] * x[i]

            # выражаем текущую переменную
            x_new[i] = (b[i] - s) / A[i, i]

        diff = np.linalg.norm(x_new - x, ord=np.inf)   # максимальное изменение
        history.append((k + 1, x_new.copy(), diff))

        if diff < eps:
            return x_new, history

        x = x_new

    return x, history


# Пример данных
A = [[10, 1, 1],
     [2, 10, 1],
     [2, 2, 10]]

b = [12, 13, 14]

x, hist = jacobi_method(A, b)

print("Решение системы:", x)
print("Первые 5 шагов:")
for row in hist[:5]:
    print(row)

Решение системы: [1.00000007 1.00000008 1.0000001 ]
Первые 5 шагов:
(1, array([1.2, 1.3, 1.4]), np.float64(1.4))
(2, array([0.93, 0.92, 0.9 ]), np.float64(0.4999999999999999))
(3, array([1.018, 1.024, 1.03 ]), np.float64(0.13))
(4, array([0.9946, 0.9934, 0.9916]), np.float64(0.03839999999999999))
(5, array([1.0015 , 1.00192, 1.0024 ]), np.float64(0.010800000000000143))


В методе простой итерации для систем новое приближение рассчитывается из предыдущего.
На каждом шаге каждая переменная вычисляется отдельно через старые значения остальных переменных.
Процесс продолжается, пока изменения между шагами не станут достаточно малы.

2.4. Метод Зейделя

In [10]:
def seidel_method(A, b, eps=1e-6, max_iter=1000):
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    n = len(b)

    x = np.zeros(n)   # начальное приближение
    history = []

    for k in range(max_iter):
        x_old = x.copy()   # сохраняем старое приближение

        for i in range(n):
            # используем уже обновленные значения слева
            s1 = np.dot(A[i, :i], x[:i])

            # а справа пока старые значения
            s2 = np.dot(A[i, i + 1:], x_old[i + 1:])

            # пересчитываем текущую переменную
            x[i] = (b[i] - s1 - s2) / A[i, i]

        diff = np.linalg.norm(x - x_old, ord=np.inf)
        history.append((k + 1, x.copy(), diff))

        if diff < eps:
            return x, history

    return x, history


# Пример данных
A = [[10, 1, 1],
     [2, 10, 1],
     [2, 2, 10]]

b = [12, 13, 14]

x, hist = seidel_method(A, b)

print("Решение системы:", x)
print("Первые 5 шагов:")
for row in hist[:5]:
    print(row)

Решение системы: [1.00000002 0.99999998 1.        ]
Первые 5 шагов:
(1, array([1.2  , 1.06 , 0.948]), np.float64(1.2))
(2, array([0.9992  , 1.00536 , 0.999088]), np.float64(0.20079999999999987))
(3, array([0.9995552 , 1.00018016, 1.00005293]), np.float64(0.005179840000000047))
(4, array([0.99997669, 0.99999937, 1.00000479]), np.float64(0.0004214912000000126))
(5, array([0.99999958, 0.9999996 , 1.00000016]), np.float64(2.2893107200050444e-05))


Метод Зейделя похож на метод простой итерации, но работает немного эффективнее.
Здесь новые найденные значения сразу используются в текущем шаге.
Из-за этого решение часто находится быстрее, чем в методе Якоби.

3.1. Метод простой итерации для системы нелинейных уравнений

In [11]:
def nonlinear_simple_iteration(phi1, phi2, x0, y0, eps=1e-6, max_iter=1000):
    history = []
    x, y = x0, y0   # начальное приближение

    for i in range(max_iter):
        x_new = phi1(x, y)   # новое значение x
        y_new = phi2(x, y)   # новое значение y

        # берём максимальное изменение из двух переменных
        diff = max(abs(x_new - x), abs(y_new - y))
        history.append((i + 1, x, y, x_new, y_new, diff))

        if diff < eps:
            return (x_new, y_new), history

        x, y = x_new, y_new   # переходим к следующему шагу

    return (x, y), history


# Пример данных
def phi1(x, y):
    return math.sqrt((y + 5) / 2)

def phi2(x, y):
    return x - 1

solution, hist = nonlinear_simple_iteration(phi1, phi2, 1.5, 0.5)

print("Решение системы:", solution)
print("Первые 5 шагов:")
for row in hist[:5]:
    print(row)

Решение системы: (1.6861406177375333, 0.6861403655686218)
Первые 5 шагов:
(1, 1.5, 0.5, 1.6583123951777, 0.5, 0.1583123951776999)
(2, 1.6583123951777, 0.5, 1.6583123951777, 0.6583123951776999, 0.1583123951776999)
(3, 1.6583123951777, 0.6583123951776999, 1.682009571194186, 0.6583123951776999, 0.023697176016486132)
(4, 1.682009571194186, 0.6583123951776999, 1.682009571194186, 0.682009571194186, 0.023697176016486132)
(5, 1.682009571194186, 0.682009571194186, 1.6855280435510687, 0.682009571194186, 0.0035184723568826293)


Этот метод применяется для системы нелинейных уравнений.
Каждое новое значение переменных вычисляется через предыдущие значения.
Если изменения обеих переменных становятся достаточно малыми, процесс завершается.

3.2. Метод Ньютона для системы нелинейных уравнений

In [12]:
def newton_system(F, J, x0, eps=1e-6, max_iter=100):
    x = np.array(x0, dtype=float)   # начальное приближение в виде вектора
    history = []

    for i in range(max_iter):
        Fx = np.array(F(x), dtype=float)   # вектор значений функций
        Jx = np.array(J(x), dtype=float)   # матрица Якоби

        # решаем систему J(x) * delta = -F(x)
        delta = np.linalg.solve(Jx, -Fx)

        # обновляем приближение
        x_new = x + delta

        diff = np.linalg.norm(delta, ord=np.inf)
        history.append((i + 1, x.copy(), Fx.copy(), delta.copy(), x_new.copy(), diff))

        if diff < eps:
            return x_new, history

        x = x_new

    return x, history


# Пример данных
def F(v):
    x, y = v
    return [
        x**2 + y**2 - 4,
        x - y - 1
    ]

def J(v):
    x, y = v
    return [
        [2*x, 2*y],
        [1, -1]
    ]

solution, hist = newton_system(F, J, [1.5, 0.5])

print("Решение системы:", solution)
print("Первые 5 шагов:")
for row in hist[:5]:
    print(row)

Решение системы: [1.82287566 0.82287566]
Первые 5 шагов:
(1, array([1.5, 0.5]), array([-1.5,  0. ]), array([0.375, 0.375]), array([1.875, 0.875]), np.float64(0.375))
(2, array([1.875, 0.875]), array([0.28125, 0.     ]), array([-0.05113636, -0.05113636]), array([1.82386364, 0.82386364]), np.float64(0.05113636363636364))
(3, array([1.82386364, 0.82386364]), array([0.00522986, 0.        ]), array([-0.00098761, -0.00098761]), array([1.82287602, 0.82287602]), np.float64(0.0009876121732345685))
(4, array([1.82287602, 0.82287602]), array([1.95075561e-06, 0.00000000e+00]), array([-3.68658055e-07, -3.68658055e-07]), array([1.82287566, 0.82287566]), np.float64(3.686580550505265e-07))


Метод Ньютона для системы использует вектор функций и матрицу Якоби.
На каждом шаге решается линейная система, которая даёт поправку к текущему приближению.
Метод считается одним из самых точных и быстрых, если начальное приближение выбрано удачно.